# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering

This notebook defines the eight numeric core features used by W05 and tests identifier, future-information, label-derived, query-coverage, and privacy leakage risks. It is deliberately self-contained so Colab can open it as a valid notebook without requiring a warehouse secret.

## 1. Build the feature vector

The W05 clustering vector has eight numeric features: search demand, content length, age/freshness, search visibility, and observed engagement. IDs remain identifiers only. Query breadth and future/trend labels are not part of the clustering vector.

Missing values are handled on a working copy. `avg_position_90d = 0` is treated as missing because the source convention uses zero for no position data. Volume/count features are `log1p` transformed, then the vector is RobustScaled.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

ROOT = Path("../..")
CANDIDATES = [
    ROOT / "data" / "raw" / "content_refresh_anonymized.csv",
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../outputs/content_archetypes_clustered.parquet"),
    Path("../outputs/content_level_model_dataset.parquet"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Data not found. In Colab, place content_refresh_anonymized.csv under data/raw/ or run W05 first."
    )

if DATA_PATH.suffix.lower() == ".csv":
    model_df = pd.read_csv(DATA_PATH)
else:
    model_df = pd.read_parquet(DATA_PATH)

core_features = [
    "search_volume",
    "word_count",
    "content_age_days",
    "days_since_update",
    "impressions_90d",
    "ctr_90d",
    "avg_position_90d",
    "engagement_rate",
]

id_candidates = ["client_hash_id", "client_id"]
content_id_candidates = ["content_hash_id", "content_id"]
required = [c for c in core_features if c in model_df.columns]
missing = [c for c in core_features if c not in model_df.columns]

if missing:
    raise ValueError(f"Missing required core features: {missing}")

X_raw = model_df[core_features].copy()
zero_position_count = int((X_raw["avg_position_90d"] == 0).sum())
X_raw["avg_position_90d"] = X_raw["avg_position_90d"].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=core_features,
    index=model_df.index,
)

log_features = ["search_volume", "word_count", "impressions_90d"]
for col in log_features:
    X_imputed[col] = np.log1p(X_imputed[col].clip(lower=0))

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_imputed)
feature_matrix = pd.DataFrame(X_scaled, columns=core_features, index=model_df.index)

print("Loaded:", DATA_PATH.resolve())
print("Rows:", len(model_df))
print("Core features:", len(core_features))
print("Feature matrix:", feature_matrix.shape)
print("avg_position_90d zero -> missing conversions:", zero_position_count)
print("Remaining missing after imputation:", int(X_imputed.isna().sum().sum()))

## 2. Feature notes (meaning, missing, categorical, available-when?)

Each feature is numeric. For this unsupervised task, `available-when?` means the value is available within the analysis snapshot used to describe the current content portfolio. It is not a future outcome label.

In [ ]:
feature_notes = pd.DataFrame([
    {"feature":"search_volume","meaning":"Observed search-demand signal","missing":"Median","categorical":False,"available_in_snapshot":True},
    {"feature":"word_count","meaning":"Content length in words","missing":"Median","categorical":False,"available_in_snapshot":True},
    {"feature":"content_age_days","meaning":"Content age at snapshot","missing":"Median","categorical":False,"available_in_snapshot":True},
    {"feature":"days_since_update","meaning":"Elapsed days since recorded update","missing":"Median","categorical":False,"available_in_snapshot":True},
    {"feature":"impressions_90d","meaning":"Observed search impressions over 90 days","missing":"Median + log1p","categorical":False,"available_in_snapshot":True},
    {"feature":"ctr_90d","meaning":"Observed click-through rate over 90 days","missing":"Median","categorical":False,"available_in_snapshot":True},
    {"feature":"avg_position_90d","meaning":"Observed average search position; zero means no position data","missing":"0→NaN, then median","categorical":False,"available_in_snapshot":True},
    {"feature":"engagement_rate","meaning":"Observed engaged-session share","missing":"Median","categorical":False,"available_in_snapshot":True},
])
display(feature_notes)

## 3. The leakage hunt

The tests below attack the vector directly. For clustering, leakage means allowing a field to encode identity, future information, outcome/label information, or data-coverage artifacts that are outside the intended archetype definition.

In [ ]:
future_keywords = ["future", "trend", "decline", "outcome", "label", "target"]
query_columns = [c for c in model_df.columns if "query" in c.lower()]
privacy_columns = [
    c for c in model_df.columns
    if any(k in c.lower() for k in ["client_name", "company", "domain", "brand", "url"])
]

identifier_columns = [c for c in model_df.columns if c in id_candidates + content_id_candidates]
future_like_columns = [c for c in model_df.columns if any(k in c.lower() for k in future_keywords)]

leakage_audit = pd.DataFrame([
    {"risk":"Identifier leakage","found":identifier_columns,"in_core":sorted(set(identifier_columns)&set(core_features)),"status":"PASS" if not (set(identifier_columns)&set(core_features)) else "FAIL"},
    {"risk":"Future/trend/label-like fields","found":future_like_columns,"in_core":sorted(set(future_like_columns)&set(core_features)),"status":"PASS" if not (set(future_like_columns)&set(core_features)) else "FAIL"},
    {"risk":"Query-breadth/data-coverage feature","found":query_columns,"in_core":sorted(set(query_columns)&set(core_features)),"status":"PASS" if not (set(query_columns)&set(core_features)) else "FAIL"},
    {"risk":"Client/URL privacy exposure","found":privacy_columns,"in_core":sorted(set(privacy_columns)&set(core_features)),"status":"PASS" if not (set(privacy_columns)&set(core_features)) else "FAIL"},
])

display(leakage_audit)

### Leakage conclusion

The core vector contains eight snapshot-level metric features only. IDs are not used as model inputs. Query breadth remains outside the core feature set because its missingness can represent data coverage rather than a substantive content archetype. Future/trend/label-like fields are excluded.

In [ ]:
assert "client_hash_id" not in core_features
assert "content_hash_id" not in core_features
assert "client_id" not in core_features
assert "content_id" not in core_features
assert not any(k in c.lower() for c in core_features for k in future_keywords)
assert not any("query" in c.lower() for c in core_features)
assert not any(any(k in c.lower() for k in ["client_name", "company", "domain", "brand", "url"]) for c in core_features)
assert int(X_imputed.isna().sum().sum()) == 0
print("All leakage assertions PASS.")

## 4. What I excluded and why

Excluded fields may remain available for joining or profiling, but they must not define the cluster.

In [ ]:
exclusions = pd.DataFrame([
    {"field":"client_hash_id / client_id","reason":"Identifier only; could create client-specific groups rather than content archetypes."},
    {"field":"content_hash_id / content_id","reason":"Identifier only; unique IDs do not encode the intended archetype signal."},
    {"field":"query_count_90d / query breadth","reason":"Can encode query-data coverage and produce a missingness-driven cluster."},
    {"field":"future / trend / label-like fields","reason":"Could introduce later information or outcome definitions into the current snapshot."},
    {"field":"client names / company / domain / URL","reason":"Not needed for clustering and should not appear in paper-facing analysis."},
    {"field":"provider / model metadata","reason":"Describes generation infrastructure rather than content performance."},
])
display(exclusions)

## Self-check

The checks below verify the mechanical requirements of the feature vector and leakage audit.

In [ ]:
checks = [
    ("Source data loaded", len(model_df) > 0),
    ("Exactly eight core features", len(core_features) == 8),
    ("All core features exist", all(c in model_df.columns for c in core_features)),
    ("Identifiers excluded", not any(c in core_features for c in ["client_id", "client_hash_id", "content_id", "content_hash_id"])),
    ("Future/trend/label-like fields excluded", not any(k in c.lower() for c in core_features for k in future_keywords)),
    ("Query fields excluded", not any("query" in c.lower() for c in core_features)),
    ("Zero position handled as missing", zero_position_count >= 0),
    ("Median imputation used", isinstance(imputer, SimpleImputer) and imputer.strategy == "median"),
    ("RobustScaler applied", isinstance(scaler, RobustScaler)),
    ("No missing values after imputation", int(X_imputed.isna().sum().sum()) == 0),
    ("Feature matrix rows match input", len(feature_matrix) == len(model_df)),
]
check_df = pd.DataFrame(checks, columns=["check", "passed"])
display(check_df)
if not check_df["passed"].all():
    raise AssertionError("One or more W03 feature leakage checks failed.")
print("All W03 feature leakage checks PASS.")
print("Manual: run the notebook top-to-bottom in a fresh Colab runtime.")